# Translation Benchmark — Swiss court paragraphs DE/FR/IT → EN

**Two goals in one notebook:**
1. **Speed/memory benchmark** across 4 open-source translation paths:
   - Helsinki-NLP opus-mt (3 per-language MarianMT models)
   - NLLB-200-distilled-600M (Meta)
   - NLLB-200-3.3B (Meta, bigger)
   - Qwen3-14B-AWQ via vLLM (continuous batching)
2. **Quality test**: a 10-query × 10-candidate eval set (1 gold + 9 close-but-non-gold per val query). After translating all 100 paragraphs with each tool, we run BM25 between each English query and the 10 English-translated candidates per query, and compute MRR / Hit@1 / Hit@3. Better translation → gold ranks higher among close near-misses.

Language is detected with `langid` (no `language` column in `court_considerations.csv`).

Models load sequentially; memory released between tools.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q transformers sentencepiece accelerate pandas pyarrow
!pip install -q langid rank_bm25
!pip install -q vllm autoawq
import torch, time, json, gc, os, re
import pandas as pd
from pathlib import Path

print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
print('Total VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Cell 2 — Mount Drive & inspect schemas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/swiss_law/data')
COURT_CSV = DATA_DIR / 'court_considerations.csv'
VAL_CSV   = DATA_DIR / 'val.csv'

OUT_DIR = Path('/content/drive/MyDrive/swiss_law/translation_benchmark_2026-05-22')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Quick schema check — adjust column names below if these don't match
_court_head = pd.read_csv(COURT_CSV, nrows=3, low_memory=False)
_val_head   = pd.read_csv(VAL_CSV, nrows=3)
print('court_considerations columns:', list(_court_head.columns))
print('val columns:', list(_val_head.columns))
print('\nval sample:')
print(_val_head.head().to_string())

## Cell 3 — Load full court corpus & val, detect language with `langid`

If your column names differ, edit `TEXT_COL`, `CIT_COL`, `QUERY_COL`, `GOLD_COL` at the top of this cell.

In [ ]:
# === Adjust if needed ===
TEXT_COL  = 'text'
CIT_COL   = 'citation'
QUERY_COL = 'query'
GOLD_COL  = 'gold_citations'
QID_COL   = 'query_id'
# =========================

import langid
langid.set_languages(['de', 'fr', 'it'])

def detect_lang(text):
    if not isinstance(text, str) or len(text) < 20:
        return 'unknown'
    lang, _ = langid.classify(text[:500])
    return lang

court_df = pd.read_csv(COURT_CSV, low_memory=False)
court_df[TEXT_COL] = court_df[TEXT_COL].astype(str)
print('court rows:', len(court_df))

val_df = pd.read_csv(VAL_CSV)
print('val queries:', len(val_df))
print('val columns:', list(val_df.columns))

## Cell 4 — Build the eval set: 10 val queries × 10 candidates each

For each val query, take 1 gold court citation + 9 close-but-non-gold candidates.

**Close-non-gold ranking** (V1 research-aligned):
1. **Same case_id**, different E. section (the hardest near-miss — e.g., `1B_28/2022 E. 4` vs gold `1B_28/2022 E. 4.1`). Up to 5.
2. **Same statute** mentioned in text — fill remaining slots.

Result: `eval_set` (100 rows: 10 gold + 90 close non-gold) with columns `query_id`, `query`, `citation`, `text`, `is_gold`, `language`.

In [ ]:
# Helper: extract case_id (everything before ' E. ')
def case_id_of(citation):
    if not isinstance(citation, str): return None
    m = re.match(r'^(.+?)\s+E\.\s', citation)
    return m.group(1).strip() if m else citation.strip()

# Helper: extract first statute reference from text (Art./art. NUMBER CODE)
STATUTE_RE = re.compile(r'\b[Aa]rt(?:icle)?\.?\s+\d+[a-z]?(?:\s+(?:Abs|al|cpv)\.?\s+\d+)?(?:\s+(?:lit|let|lett)\.?\s+[a-z])?\s+([A-Z]{2,6})\b')

def first_statute(text):
    m = STATUTE_RE.search(text)
    return m.group(0) if m else None

# Build a set of all val gold citations (so we never pick a 'non-gold' that is gold for another query)
all_gold = set()
for _, vq in val_df.iterrows():
    g = vq[GOLD_COL]
    if isinstance(g, str):
        for c in re.split(r'[;,]', g):
            c = c.strip()
            if c: all_gold.add(c)
print('Total gold citations across val:', len(all_gold))

# Index court_df by citation for fast lookup
court_by_cit = court_df.set_index(CIT_COL)

eval_rows = []
for _, vq in val_df.iterrows():
    qid = vq[QID_COL]
    qtext = vq[QUERY_COL]
    gold_list = [c.strip() for c in re.split(r'[;,]', str(vq[GOLD_COL])) if c.strip()]

    # 1. Pick 1 gold COURT citation (must exist in court_df)
    gold_cit = None; gold_row = None
    for cit in gold_list:
        if cit in court_by_cit.index:
            row = court_by_cit.loc[cit]
            if isinstance(row, pd.DataFrame): row = row.iloc[0]
            gold_cit = cit; gold_row = row
            break
    if gold_cit is None:
        print(f'  {qid}: NO court gold found in corpus — skipping')
        continue

    eval_rows.append({
        'query_id': qid, 'query': qtext,
        'citation': gold_cit, 'text': gold_row[TEXT_COL],
        'is_gold': True,
    })

    # 2. Close non-gold pool A: same case_id, different E.
    cid = case_id_of(gold_cit)
    same_case = court_df[court_df[CIT_COL].str.startswith(cid + ' ', na=False) & (court_df[CIT_COL] != gold_cit)]
    same_case = same_case[~same_case[CIT_COL].isin(all_gold)]
    same_case = same_case[same_case[TEXT_COL].str.len() >= 200]

    picks = same_case.sample(min(5, len(same_case)), random_state=42) if len(same_case) > 0 else same_case.iloc[:0]

    # 3. Fill remaining with same-statute paragraphs
    need = 9 - len(picks)
    if need > 0:
        statute = first_statute(gold_row[TEXT_COL])
        used_cits = set(picks[CIT_COL].tolist()) | all_gold | {gold_cit}
        if statute:
            # Use a relatively rare statute fragment to avoid scanning all rows
            mask = court_df[TEXT_COL].str.contains(re.escape(statute), regex=True, na=False)
            same_stat = court_df[mask & ~court_df[CIT_COL].isin(used_cits) & (court_df[TEXT_COL].str.len() >= 200)]
            if len(same_stat) > 0:
                extra = same_stat.sample(min(need, len(same_stat)), random_state=42)
                picks = pd.concat([picks, extra], ignore_index=True)
        need = 9 - len(picks)
        if need > 0:
            # last-resort fill: random federal court paragraphs
            rest = court_df[~court_df[CIT_COL].isin(all_gold) & ~court_df[CIT_COL].isin(picks[CIT_COL])]
            extra = rest.sample(need, random_state=42)
            picks = pd.concat([picks, extra], ignore_index=True)

    picks = picks.head(9)
    for _, p in picks.iterrows():
        eval_rows.append({
            'query_id': qid, 'query': qtext,
            'citation': p[CIT_COL], 'text': p[TEXT_COL],
            'is_gold': False,
        })
    print(f'  {qid}: 1 gold + {len(picks)} close non-gold (case-matches: {min(5, len(same_case))})')

samples = pd.DataFrame(eval_rows).reset_index(drop=True)
samples['language'] = samples['text'].apply(detect_lang)
samples = samples[samples['language'] != 'unknown'].reset_index(drop=True)

samples.to_csv(OUT_DIR / 'input_samples.csv', index=False)
print('\nFinal eval set:', len(samples), 'rows')
print('Languages:', samples['language'].value_counts().to_dict())
print('Gold count:', samples['is_gold'].sum())
print('Mean text length (chars):', round(samples['text'].str.len().mean()))

## Cell 5 — Benchmark utilities (memory, metrics, output saving)

In [ ]:
def reset_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

def vram_gb():
    return round(torch.cuda.max_memory_allocated() / 1e9, 2)

def save_metrics_and_translations(name, samples, translations, in_tokens, out_tokens, elapsed, peak_mem):
    out_df = samples.copy()
    out_df['translation'] = translations
    out_df.to_csv(OUT_DIR / f'{name}_translations.csv', index=False)
    metrics = {
        'tool': name,
        'n_samples': len(samples),
        'elapsed_sec': round(elapsed, 2),
        'input_tokens_total': in_tokens,
        'output_tokens_total': out_tokens,
        'input_tokens_per_sample': round(in_tokens / len(samples), 1),
        'output_tokens_per_sample': round(out_tokens / len(samples), 1),
        'output_tokens_per_sec': round(out_tokens / elapsed, 1),
        'paragraphs_per_sec': round(len(samples) / elapsed, 2),
        'peak_vram_gb': peak_mem,
    }
    with open(OUT_DIR / f'{name}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))
    return metrics

ALL_METRICS = []

## Cell 6 — Tool 1: Helsinki-NLP opus-mt (3 per-language MarianMT models)

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

reset_gpu()
opus_tok, opus_mdl = {}, {}
for lang in ['de', 'fr', 'it']:
    n = f'Helsinki-NLP/opus-mt-{lang}-en'
    opus_tok[lang] = MarianTokenizer.from_pretrained(n)
    opus_mdl[lang] = MarianMTModel.from_pretrained(n).to('cuda').half().eval()
print('opus-mt loaded, VRAM:', vram_gb(), 'GB')

BATCH = 128
translations = [''] * len(samples)
in_tokens = out_tokens = 0

t0 = time.time()
for lang in ['de', 'fr', 'it']:
    idxs = [i for i, l in enumerate(samples['language']) if l == lang]
    if not idxs: continue
    tok, mdl = opus_tok[lang], opus_mdl[lang]
    for s in range(0, len(idxs), BATCH):
        b_idxs = idxs[s:s+BATCH]
        b_texts = [samples['text'].iloc[j] for j in b_idxs]
        inp = tok(b_texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
        in_tokens += int(inp['attention_mask'].sum())
        with torch.no_grad():
            gen = mdl.generate(**inp, max_length=512, num_beams=1, do_sample=False)
        out_tokens += int((gen != tok.pad_token_id).sum())
        for k, d in zip(b_idxs, tok.batch_decode(gen, skip_special_tokens=True)):
            translations[k] = d
torch.cuda.synchronize()
elapsed = time.time() - t0
peak = vram_gb()

ALL_METRICS.append(save_metrics_and_translations('opus-mt', samples, translations, in_tokens, out_tokens, elapsed, peak))

for lang in list(opus_mdl.keys()): del opus_mdl[lang], opus_tok[lang]
del opus_mdl, opus_tok
reset_gpu()
print('After cleanup VRAM (alloc):', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

## Cell 7 — Tool 2: NLLB-200-distilled-600M

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

reset_gpu()
name = 'facebook/nllb-200-distilled-600M'
tok = AutoTokenizer.from_pretrained(name)
mdl = AutoModelForSeq2SeqLM.from_pretrained(name, torch_dtype=torch.float16).to('cuda').eval()
print('NLLB-600M loaded, VRAM:', vram_gb(), 'GB')

LANG_MAP = {'de': 'deu_Latn', 'fr': 'fra_Latn', 'it': 'ita_Latn'}
TGT = tok.convert_tokens_to_ids('eng_Latn')
BATCH = 128

translations = [''] * len(samples)
in_tokens = out_tokens = 0

t0 = time.time()
for lang in ['de', 'fr', 'it']:
    idxs = [i for i, l in enumerate(samples['language']) if l == lang]
    if not idxs: continue
    tok.src_lang = LANG_MAP[lang]
    for s in range(0, len(idxs), BATCH):
        b_idxs = idxs[s:s+BATCH]
        b_texts = [samples['text'].iloc[j] for j in b_idxs]
        inp = tok(b_texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
        in_tokens += int(inp['attention_mask'].sum())
        with torch.no_grad():
            gen = mdl.generate(**inp, forced_bos_token_id=TGT, max_length=512, num_beams=1, do_sample=False)
        out_tokens += int((gen != tok.pad_token_id).sum())
        for k, d in zip(b_idxs, tok.batch_decode(gen, skip_special_tokens=True)):
            translations[k] = d
torch.cuda.synchronize()
elapsed = time.time() - t0
peak = vram_gb()

ALL_METRICS.append(save_metrics_and_translations('nllb-200-600M', samples, translations, in_tokens, out_tokens, elapsed, peak))
del mdl, tok
reset_gpu()
print('After cleanup VRAM (alloc):', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

## Cell 8 — Tool 3: NLLB-200-3.3B

In [ ]:
reset_gpu()
name = 'facebook/nllb-200-3.3B'
tok = AutoTokenizer.from_pretrained(name)
mdl = AutoModelForSeq2SeqLM.from_pretrained(name, torch_dtype=torch.float16).to('cuda').eval()
print('NLLB-3.3B loaded, VRAM:', vram_gb(), 'GB')

LANG_MAP = {'de': 'deu_Latn', 'fr': 'fra_Latn', 'it': 'ita_Latn'}
TGT = tok.convert_tokens_to_ids('eng_Latn')
BATCH = 64

translations = [''] * len(samples)
in_tokens = out_tokens = 0

t0 = time.time()
for lang in ['de', 'fr', 'it']:
    idxs = [i for i, l in enumerate(samples['language']) if l == lang]
    if not idxs: continue
    tok.src_lang = LANG_MAP[lang]
    for s in range(0, len(idxs), BATCH):
        b_idxs = idxs[s:s+BATCH]
        b_texts = [samples['text'].iloc[j] for j in b_idxs]
        inp = tok(b_texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
        in_tokens += int(inp['attention_mask'].sum())
        with torch.no_grad():
            gen = mdl.generate(**inp, forced_bos_token_id=TGT, max_length=512, num_beams=1, do_sample=False)
        out_tokens += int((gen != tok.pad_token_id).sum())
        for k, d in zip(b_idxs, tok.batch_decode(gen, skip_special_tokens=True)):
            translations[k] = d
torch.cuda.synchronize()
elapsed = time.time() - t0
peak = vram_gb()

ALL_METRICS.append(save_metrics_and_translations('nllb-200-3.3B', samples, translations, in_tokens, out_tokens, elapsed, peak))
del mdl, tok
reset_gpu()
print('After cleanup VRAM (alloc):', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

## Cell 9 — Tool 4: Qwen3-14B-AWQ via vLLM (continuous batching)

If `Qwen/Qwen3-14B-AWQ` 404s, swap to `cpatonn/Qwen3-14B-AWQ` (community AWQ) or `Qwen/Qwen3-14B` (BF16, fits Blackwell easily). `enable_thinking=False` is critical.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

reset_gpu()
MODEL_NAME = 'Qwen/Qwen3-14B-AWQ'

llm = LLM(
    model=MODEL_NAME,
    quantization='awq_marlin',
    max_model_len=4096,
    gpu_memory_utilization=0.85,
    dtype='float16',
    trust_remote_code=True,
)
qwen_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print('Qwen3-14B-AWQ loaded, VRAM:', vram_gb(), 'GB')

LANG_NAME = {'de': 'German', 'fr': 'French', 'it': 'Italian'}

def build_prompt(text, lang):
    msgs = [
        {'role': 'system', 'content': 'You are a professional legal translator. Output ONLY the English translation. No explanation, no preamble.'},
        {'role': 'user',   'content': f'Translate this {LANG_NAME.get(lang, "Swiss legal")} paragraph to English:\n\n{text}'},
    ]
    return qwen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

prompts = [build_prompt(t, l) for t, l in zip(samples['text'], samples['language'])]
sampling = SamplingParams(temperature=0.0, max_tokens=1024)

t0 = time.time()
outputs = llm.generate(prompts, sampling)
torch.cuda.synchronize()
elapsed = time.time() - t0
peak = vram_gb()

translations = [o.outputs[0].text.strip() for o in outputs]
in_tokens = sum(len(o.prompt_token_ids) for o in outputs)
out_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)

ALL_METRICS.append(save_metrics_and_translations('qwen3-14B-awq', samples, translations, in_tokens, out_tokens, elapsed, peak))
del llm, qwen_tok
reset_gpu()
print('After cleanup VRAM (alloc):', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

## Cell 10 — Quality test: BM25 ranking on translations

For each tool, for each of the 10 val queries with a gold candidate in the set:
1. Build BM25 over the 10 English-translated candidates (1 gold + 9 close non-gold).
2. Score the English query against them.
3. Find the rank of the gold (1 = best).

**Metrics**: MRR, Hit@1, Hit@3, Hit@5. Higher = the translation preserves enough doctrinal signal that lexical matching can pick gold out of the near-miss pool.

This is the directly relevant metric for whether translation quality matters to the retrieval pipeline.

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\b[a-zA-Z]{3,}\b", str(text).lower())

quality = []
for tool in [m['tool'] for m in ALL_METRICS]:
    tdf = pd.read_csv(OUT_DIR / f'{tool}_translations.csv')
    mrr, h1, h3, h5, n = 0.0, 0, 0, 0, 0
    gold_ranks = []

    for qid in tdf['query_id'].unique():
        qrows = tdf[tdf['query_id'] == qid].reset_index(drop=True)
        if not qrows['is_gold'].any() or len(qrows) < 2:
            continue
        corpus = [tokenize(t) for t in qrows['translation']]
        bm25 = BM25Okapi(corpus)
        qtext = qrows['query'].iloc[0]
        scores = bm25.get_scores(tokenize(qtext))

        order = sorted(range(len(scores)), key=lambda i: -scores[i])
        gold_local_idx = qrows.index[qrows['is_gold']].tolist()[0]
        rank = order.index(gold_local_idx) + 1
        gold_ranks.append({'query_id': qid, 'gold_rank': rank, 'pool_size': len(qrows)})

        mrr += 1.0 / rank
        if rank == 1: h1 += 1
        if rank <= 3: h3 += 1
        if rank <= 5: h5 += 1
        n += 1

    quality.append({
        'tool': tool,
        'n_queries_with_gold': n,
        'MRR': round(mrr / max(n, 1), 3),
        'Hit@1': round(h1 / max(n, 1), 3),
        'Hit@3': round(h3 / max(n, 1), 3),
        'Hit@5': round(h5 / max(n, 1), 3),
    })
    pd.DataFrame(gold_ranks).to_csv(OUT_DIR / f'{tool}_gold_ranks.csv', index=False)

quality_df = pd.DataFrame(quality)
quality_df.to_csv(OUT_DIR / 'quality_summary.csv', index=False)
print(quality_df.to_string(index=False))

## Cell 11 — Combined summary (speed + quality + corpus projection)

In [ ]:
speed_df = pd.DataFrame(ALL_METRICS)
FULL_N = 2_476_315
speed_df['full_corpus_hours'] = (speed_df['elapsed_sec'] / speed_df['n_samples'] * FULL_N / 3600).round(1)
speed_df['full_corpus_output_tokens_M'] = (speed_df['output_tokens_total'] / speed_df['n_samples'] * FULL_N / 1e6).round(1)

combined = speed_df.merge(quality_df, on='tool', how='left')
combined.to_csv(OUT_DIR / 'benchmark_summary.csv', index=False)
print(combined.to_string(index=False))

print('\n=== Files in', OUT_DIR, '===')
for p in sorted(OUT_DIR.iterdir()):
    print(' ', p.name, ' (', round(p.stat().st_size / 1024, 1), 'KB)')

## Cell 12 — Spot-check: side-by-side translation of one gold paragraph per language

In [ ]:
tool_dfs = {m['tool']: pd.read_csv(OUT_DIR / f"{m['tool']}_translations.csv") for m in ALL_METRICS}

for lang in ['de', 'fr', 'it']:
    cand = samples[(samples['language'] == lang) & (samples['is_gold'])]
    if len(cand) == 0: continue
    row = cand.iloc[0]
    print('\n' + '='*100)
    print(f'LANG: {lang.upper()}  |  QUERY_ID: {row["query_id"]}  |  CITATION: {row["citation"]}  |  GOLD: True')
    print('='*100)
    print('[QUERY]')
    print(row['query'][:400])
    print('\n[SOURCE TEXT]')
    print(row['text'][:500])
    for name, tdf in tool_dfs.items():
        match = tdf[(tdf['query_id'] == row['query_id']) & (tdf['citation'] == row['citation'])]
        if len(match):
            print(f'\n[{name.upper()}]')
            print(str(match['translation'].iloc[0])[:500])